In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("cleaned_retail.csv")
rfm = pd.read_csv("rfm_table.csv")

data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"])

print(data.shape)
print(rfm.shape)
print(rfm.head())

(804824, 8)
(5855, 4)
   Customer ID  Recency  Frequency  Monetary
0      15712.0        9         12   3472.41
1      12931.0       21         57  92347.34
2      17611.0        7         28   8321.98
3      16620.0        3          6   1758.93
4      13381.0       31         12   8224.37


Define the churn label

In [3]:
rfm["Churned"] = (rfm["Recency"] > 90).astype(int)

print(rfm["Churned"].value_counts())
print(rfm["Churned"].value_counts(normalize=True) * 100)


Churned
1    2968
0    2887
Name: count, dtype: int64
Churned
1    50.691716
0    49.308284
Name: proportion, dtype: float64


## Churn Label Definition

Defined churn as: no purchase within the last 90 days, based on the Recency feature from the RFM table (Module A). This threshold was chosen deliberately — it sits close to the dataset's median Recency (95 days, from Day 2 analysis), producing a genuinely balanced target (50.7% churned, 49.3% active) rather than a heavily skewed one. A shorter threshold (e.g., 30 days) would have classified most customers as "churned," making the label far less useful for a real business decision.

## Step 3: Train/Test Split (with Leakage Check)

Excluded Recency from the feature set, since it was used to define the Churned label itself (Recency > 90 = Churned) — including it would let the model simply learn back the labeling rule rather than genuinely predict churn. Used a stratified split to preserve the ~50/50 churn balance in both train and test sets.

In [4]:
from sklearn.model_selection import train_test_split

feature_cols = ["Frequency", "Monetary"]
X = rfm[feature_cols]
y = rfm["Churned"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Train churn rate:", y_train.mean())
print("Test churn rate:", y_test.mean())


Train: (4684, 2) Test: (1171, 2)
Train churn rate: 0.5068317677198976
Test churn rate: 0.5072587532023911


## Step 4: Baseline Model — Logistic Regression

Building a simple baseline first, before the more complex model, to establish a minimum performance bar. Using the same LogisticRegression tool from the Uplift Module, now applied to a standard classification task instead of a two-model uplift comparison.

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

baseline_model = LogisticRegression(max_iter=1000, random_state=42)
baseline_model.fit(X_train, y_train)

baseline_preds = baseline_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, baseline_preds))
print("Precision:", precision_score(y_test, baseline_preds))
print("Recall:", recall_score(y_test, baseline_preds))
print("F1:", f1_score(y_test, baseline_preds))

Accuracy: 0.6900085397096499
Precision: 0.6542056074766355
Recall: 0.8249158249158249
F1: 0.7297096053611318


## Baseline Results

Logistic Regression on Frequency + Monetary alone: Accuracy 69.0%, Precision 65.4%, Recall 82.5%, F1 0.73. The model is biased toward flagging customers as churned (higher recall than precision) — a reasonable tradeoff for this use case, since missing an actual at-risk customer is typically costlier than a false alarm.

## Step 5: XGBoost Model

Building the primary model — XGBoost, a gradient-boosted tree ensemble — to see if it captures more signal than the linear baseline. Same features, same train/test split, for a fair comparison.

In [6]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42
)
xgb_model.fit(X_train, y_train)
xgb_preds = xgb_model.predict(X_test)


print("Accuracy:", accuracy_score(y_test, xgb_preds))
print("Precision:", precision_score(y_test, xgb_preds))
print("Recall:", recall_score(y_test, xgb_preds))
print("F1:", f1_score(y_test, xgb_preds))

Accuracy: 0.6942783945345858
Precision: 0.6755952380952381
Recall: 0.7643097643097643
F1: 0.717219589257504


## Step 6: Feature Engineering — New Features & Interactions

Adding features beyond the original Frequency/Monetary, without reintroducing Recency leakage:
- **AvgOrderValue** (Monetary / Frequency) — spend per transaction, a genuinely different signal than either raw feature alone
- **Frequency_log, Monetary_log** — log-transformed versions, since both are right-skewed (established back in Module A)
- **Tenure** — days between a customer's first and last purchase, computed from the raw transaction data. This reflects how long someone has been a customer, distinct from Recency (which reflects how recently they last bought) — not leakage, since it doesn't encode "distance from the dataset's end date."
- **Frequency × Monetary interaction** — lets the model see combined high-frequency/high-spend behavior directly, rather than requiring it to learn the interaction itself

In [7]:
import numpy as np

# Tenure: days between first and last purchase, per customer
tenure = data.groupby("Customer ID")["InvoiceDate"].agg(["min", "max"])
tenure["Tenure"] = (tenure["max"] - tenure["min"]).dt.days
tenure = tenure[["Tenure"]].reset_index()

rfm = rfm.merge(tenure, on="Customer ID", how="left")

rfm["AvgOrderValue"] = rfm["Monetary"] / rfm["Frequency"]
rfm["Frequency_log"] = np.log1p(rfm["Frequency"])
rfm["Monetary_log"] = np.log1p(rfm["Monetary"])
rfm["Freq_Monetary_Interaction"] = rfm["Frequency"] * rfm["Monetary"]

print(
    rfm[
        [
            "Customer ID",
            "Frequency",
            "Monetary",
            "Tenure",
            "AvgOrderValue",
            "Frequency_log",
            "Monetary_log",
            "Freq_Monetary_Interaction",
        ]
    ].head()
)


   Customer ID  Frequency  Monetary  Tenure  AvgOrderValue  Frequency_log  \
0      15712.0         12   3472.41     729     289.367500       2.564949   
1      12931.0         57  92347.34     717    1620.128772       4.060443   
2      17611.0         28   8321.98     731     297.213571       3.367296   
3      16620.0          6   1758.93     734     293.155000       1.945910   
4      13381.0         12   8224.37     706     685.364167       2.564949   

   Monetary_log  Freq_Monetary_Interaction  
0      8.152892                   41668.92  
1     11.433323                 5263798.38  
2      9.026776                  233015.44  
3      7.473029                   10553.58  
4      9.014979                   98692.44  


## Step 7: Rebuild Train/Test Split with Expanded Features

In [8]:
feature_cols_v2 = [
    "Frequency",
    "Monetary",
    "Tenure",
    "AvgOrderValue",
    "Frequency_log",
    "Monetary_log",
    "Freq_Monetary_Interaction",
]

X2 = rfm[feature_cols_v2]
y2 = rfm["Churned"]

X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, random_state=42, stratify=y2
)

print("Train:", X2_train.shape, "Test:", X2_test.shape)


Train: (4684, 7) Test: (1171, 7)


## Step 8: Three-Model Comparison on Expanded Features

Re-running Logistic Regression and XGBoost on the new 7-feature set, plus adding a Multi-Layer Perceptron (MLPClassifier) — a genuine neural network — using scikit-learn's implementation rather than TensorFlow, to avoid the NumPy/TensorFlow environment conflict encountered in the Forecasting Module.

In [9]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler

# Scale features - neural networks and logistic regression both benefit from this,
# especially now that Freq_Monetary_Interaction has a very different scale than the rest
scaler = StandardScaler()
X2_train_scaled = scaler.fit_transform(X2_train)
X2_test_scaled = scaler.transform(X2_test)

# Logistic Regression (v2)
lr_v2 = LogisticRegression(max_iter=1000, random_state=42)
lr_v2.fit(X2_train_scaled, y2_train)
lr_v2_preds = lr_v2.predict(X2_test_scaled)

# XGBoost (v2) - tree-based, doesn't need scaling, use original unscaled features
xgb_v2 = XGBClassifier(
    n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42
)
xgb_v2.fit(X2_train, y2_train)
xgb_v2_preds = xgb_v2.predict(X2_test)

# Neural Network (MLP)
mlp = MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=1000, random_state=42)
mlp.fit(X2_train_scaled, y2_train)
mlp_preds = mlp.predict(X2_test_scaled)

for name, preds in [
    ("Logistic Regression v2", lr_v2_preds),
    ("XGBoost v2", xgb_v2_preds),
    ("Neural Network (MLP)", mlp_preds),
]:
    print(f"\n{name}")
    print("Accuracy:", accuracy_score(y2_test, preds))
    print("Precision:", precision_score(y2_test, preds))
    print("Recall:", recall_score(y2_test, preds))
    print("F1:", f1_score(y2_test, preds))



Logistic Regression v2
Accuracy: 0.7096498719043552
Precision: 0.699685534591195
Recall: 0.7491582491582491
F1: 0.7235772357723578

XGBoost v2
Accuracy: 0.7190435525192144
Precision: 0.6939970717423133
Recall: 0.797979797979798
F1: 0.7423649177760376

Neural Network (MLP)
Accuracy: 0.7216054654141759
Precision: 0.6942028985507246
Recall: 0.8063973063973064
F1: 0.7461059190031153


## Step 9: Random Forest

In [10]:
from sklearn.ensemble import RandomForestClassifier

rf_v2 = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_v2.fit(X2_train, y2_train)  # tree-based, no scaling needed
rf_v2_preds = rf_v2.predict(X2_test)

print("Accuracy:", accuracy_score(y2_test, rf_v2_preds))
print("Precision:", precision_score(y2_test, rf_v2_preds))
print("Recall:", recall_score(y2_test, rf_v2_preds))
print("F1:", f1_score(y2_test, rf_v2_preds))


Accuracy: 0.7241673783091375
Precision: 0.6960926193921853
Recall: 0.8097643097643098
F1: 0.7486381322957198


## Step 10: Ensemble

Rebuilt the ensemble to include all four base models — Logistic Regression, XGBoost, Neural Network, and Random Forest — combined via soft voting.

In [11]:
from sklearn.ensemble import VotingClassifier

ensemble_v2 = VotingClassifier(
    estimators=[
        ("lr", LogisticRegression(max_iter=1000, random_state=42)),
        (
            "xgb",
            XGBClassifier(
                n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42
            ),
        ),
        (
            "mlp",
            MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=1000, random_state=42),
        ),
        ("rf", RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)),
    ],
    voting="soft",
)

ensemble_v2.fit(X2_train_scaled, y2_train)
ensemble_v2_preds = ensemble_v2.predict(X2_test_scaled)

print("Accuracy:", accuracy_score(y2_test, ensemble_v2_preds))
print("Precision:", precision_score(y2_test, ensemble_v2_preds))
print("Recall:", recall_score(y2_test, ensemble_v2_preds))
print("F1:", f1_score(y2_test, ensemble_v2_preds))


Accuracy: 0.7275832621690862
Precision: 0.7055306427503737
Recall: 0.7946127946127947
F1: 0.7474267616785432


## Final Model Selection

Adding Random Forest to the ensemble (4 models total) did not meaningfully improve over the 3-model ensemble or standalone Random Forest — Accuracy remained at 72.8%, F1 (0.747) sat between the two comparison points rather than exceeding either. This suggests Random Forest's predictions substantially overlap with the existing ensemble members, adding redundancy rather than new signal.

**Final decision:** given the standalone **Random Forest model has the best F1 (0.749) of any single model, with 81.0% recall (only the 2-feature logistic baseline recalled slightly more, 82.5%, at much lower precision), and is simpler to deploy than a 4-model ensemble**, Random Forest is selected as the production model for this system — trading a marginal amount of ensemble complexity for near-identical performance and a much simpler deployment footprint.

## Step 11: MLflow Experiment Tracking Setup

In [12]:
import mlflow
import mlflow.sklearn
import mlflow.xgboost


mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("churn_prediction")

mlflow.set_experiment("churn_prediction")

trusted_types = ["sklearn.neural_network._stochastic_optimizers.AdamOptimizer", "sklearn.tree._tree.Tree"]

models_to_log = {
    "LogisticRegression_2features": (
        LogisticRegression(max_iter=1000, random_state=42),
        X_train,
        X_test,
        y_train,
        y_test,
    ),
    "XGBoost_2features": (
        XGBClassifier(
            n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42
        ),
        X_train,
        X_test,
        y_train,
        y_test,
    ),
    "LogisticRegression_7features": (
        LogisticRegression(max_iter=1000, random_state=42),
        X2_train_scaled,
        X2_test_scaled,
        y2_train,
        y2_test,
    ),
    "XGBoost_7features": (
        XGBClassifier(
            n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42
        ),
        X2_train,
        X2_test,
        y2_train,
        y2_test,
    ),
    "NeuralNetwork_MLP": (
        MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=1000, random_state=42),
        X2_train_scaled,
        X2_test_scaled,
        y2_train,
        y2_test,
    ),
    "RandomForest": (
        RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
        X2_train,
        X2_test,
        y2_train,
        y2_test,
    ),
}

for name, (model, xtr, xte, ytr, yte) in models_to_log.items():
    with mlflow.start_run(run_name=name):
        model.fit(xtr, ytr)
        preds = model.predict(xte)

        mlflow.log_param("model_type", name)
        mlflow.log_metric("accuracy", accuracy_score(yte, preds))
        mlflow.log_metric("precision", precision_score(yte, preds))
        mlflow.log_metric("recall", recall_score(yte, preds))
        mlflow.log_metric("f1", f1_score(yte, preds))

        if "XGBoost" in name:
            mlflow.xgboost.log_model(model, "model")
        else:
            mlflow.sklearn.log_model(model, "model", skops_trusted_types=trusted_types)

print("All models logged to MLflow.")


2026/08/25 13:34:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/25 13:34:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/25 13:34:34 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/25 13:34:38 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/25 13:34:40 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/08/25 13:34:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


All models logged to MLflow.


## MLflow Tracking: Complete

Successfully logged 6 models to MLflow under the "churn_prediction" experiment: Logistic Regression and XGBoost (2-feature baseline), Logistic Regression, XGBoost, Neural Network, and Random Forest (7-feature expanded set) — each with accuracy, precision, recall, and F1 tracked, plus the actual trained model artifact saved for potential reuse. Resolved two MLflow security-related logging errors along the way (untrusted XGBoost and neural network internal types), fixed by using XGBoost's dedicated logging function and explicitly trusting the specific flagged types.

## Step 12: FastAPI Endpoint

Building a REST API to serve churn predictions. The model is retrained directly at API startup (fast on this small dataset) rather than loaded from MLflow's saved artifacts, keeping deployment simple. The MLflow experiment tracking from earlier remains a valid, separate record of the model comparison process.



## API Testing: Confirmed Working

Tested the /predict endpoint via FastAPI's interactive docs page. Confirmed input validation correctly rejects invalid data (Frequency=0 returns a clean 400 error instead of crashing), and valid input returns a real prediction with both a binary churn label and the underlying probability score (e.g., Frequency=10, Monetary=2000, Tenure=400 → 35.8% churn probability, predicted not churned).

<br>

---

# Part 2: Predicting Who Will Leave Next

Everything above uses one snapshot of the data. The churn label and the features come from the same two-year window, so the model finds customers who have **already** gone quiet.

A business needs to know who **will** stop buying. Part 2 rebuilds the problem to look forward.

<br>

<br>

## Step 13: Forward-Looking Design

The idea is to pick a cutoff date and only let the model see what happened before it.

- **Cutoff date:** 90 days before the data ends (2011-09-10)

- **Features:** built only from purchases **before** the cutoff

- **Label:** Churned = no purchase in the **90 days after** the cutoff

- **Recency:** days since the last purchase, measured at the cutoff. It is now a fair feature, because it is known on the day of prediction and the label comes from later dates.

<br>

<br>

First, the feature list.

<br>

In [ ]:
import numpy as np
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.metrics import roc_auc_score, confusion_matrix

# The 7 features from Step 6, plus Recency
fwd_cols = [
    "Frequency",
    "Monetary",
    "Tenure",
    "AvgOrderValue",
    "Frequency_log",
    "Monetary_log",
    "Freq_Monetary_Interaction",
    "Recency",
]

<br>

Next, a function that builds the features and the label for any cutoff date.

<br>

In [ ]:
def build_forward(cutoff_date, horizon_days=90):
    """Features from purchases up to cutoff_date.
    Churned = no purchase in the next horizon_days."""

    # Split the transactions into before and after the cutoff
    past = data[data["InvoiceDate"] <= cutoff_date].copy()
    window_end = cutoff_date + pd.Timedelta(days=horizon_days)
    future = data[
        (data["InvoiceDate"] > cutoff_date) & (data["InvoiceDate"] <= window_end)
    ]

    # One row per customer, using only the past
    past["Amount"] = past["Quantity"] * past["Price"]
    t = past.groupby("Customer ID").agg(
        Frequency=("Invoice", "nunique"),
        Monetary=("Amount", "sum"),
        first_purchase=("InvoiceDate", "min"),
        last_purchase=("InvoiceDate", "max"),
    )

    # Same engineered features as Step 6
    t["Tenure"] = (t["last_purchase"] - t["first_purchase"]).dt.days
    t["AvgOrderValue"] = t["Monetary"] / t["Frequency"]
    t["Frequency_log"] = np.log1p(t["Frequency"])
    t["Monetary_log"] = np.log1p(t["Monetary"].clip(lower=0))
    t["Freq_Monetary_Interaction"] = t["Frequency"] * t["Monetary"]

    # Recency is measured at the cutoff, so it is known at prediction time
    t["Recency"] = (cutoff_date - t["last_purchase"]).dt.days

    # Label: did the customer buy again after the cutoff?
    bought_again = t.index.isin(future["Customer ID"].unique())
    t["Churned"] = (~bought_again).astype(int)

    return t

<br>

Now build the dataset for the main cutoff, and one for an earlier cutoff that is used in Step 15.

<br>

In [ ]:
# Main cutoff: 90 days before the data ends
cutoff_late = data["InvoiceDate"].max() - pd.Timedelta(days=90)

# Earlier cutoff, used later for the out-of-time test
cutoff_early = cutoff_late - pd.Timedelta(days=90)

late = build_forward(cutoff_late)
early = build_forward(cutoff_early)

print("Data covers:", data["InvoiceDate"].min().date(), "to", data["InvoiceDate"].max().date())
print("Cutoff date:", cutoff_late.date())
print("Customers who bought before the cutoff:", len(late))
print(
    "Stopped buying in the next 90 days:",
    int(late["Churned"].sum()),
    f"({late['Churned'].mean() * 100:.1f}%)",
)

Data covers: 2009-12-01 to 2011-12-09
Cutoff date: 2011-09-10
Customers who bought before the cutoff: 5258
Stopped buying in the next 90 days: 2968 (56.4%)


<br>

### Forward-Looking Dataset

- **5,258 customers** bought at least once before the cutoff.

- **2,968 of them (56.4%)** did not buy again in the next 90 days.

<br>

<br>

---

## Step 14: Model Comparison on the Forward-Looking Design

The same four algorithms as before, now on the forward-looking data.

<br>

<br>

Two helper functions: one creates the four models, one calculates the scores.

<br>

In [ ]:
def make_models():
    """The same four algorithms used in Steps 8 and 9."""
    return {
        "Logistic Regression": make_pipeline(
            StandardScaler(), LogisticRegression(max_iter=1000, random_state=42)
        ),
        "XGBoost": XGBClassifier(
            n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42
        ),
        "Neural Network (MLP)": make_pipeline(
            StandardScaler(),
            MLPClassifier(hidden_layer_sizes=(32, 16), max_iter=1000, random_state=42),
        ),
        "Random Forest": RandomForestClassifier(
            n_estimators=100, max_depth=6, random_state=42
        ),
    }


def scores(y_true, model, X):
    """Accuracy, precision, recall, F1 and AUC for one model."""
    preds = model.predict(X)
    probs = model.predict_proba(X)[:, 1]
    return {
        "Accuracy": accuracy_score(y_true, preds),
        "Precision": precision_score(y_true, preds),
        "Recall": recall_score(y_true, preds),
        "F1": f1_score(y_true, preds),
        "AUC": roc_auc_score(y_true, probs),
    }

<br>

Split the customers into training and test sets.

<br>

In [ ]:
Xf_train, Xf_test, yf_train, yf_test = train_test_split(
    late[fwd_cols],
    late["Churned"],
    test_size=0.2,
    random_state=42,
    stratify=late["Churned"],
)

print("Train customers:", len(Xf_train))
print("Test customers:", len(Xf_test))
print("Always-predict-churn baseline accuracy:", round(yf_test.mean(), 3))

Train customers: 4206
Test customers: 1052
Always-predict-churn baseline accuracy: 0.565


<br>

Train all four models and compare them on the test customers.

<br>

In [ ]:
fitted = {}
rows = {}

# Train each model and score it on the held-out customers
for name, model in make_models().items():
    fitted[name] = model.fit(Xf_train, yf_train)
    rows[name] = scores(yf_test, fitted[name], Xf_test)

# Same Random Forest without Recency, to see how much Recency adds
no_recency = [c for c in fwd_cols if c != "Recency"]
rf_no_recency = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_no_recency.fit(Xf_train[no_recency], yf_train)
rows["Random Forest without Recency"] = scores(yf_test, rf_no_recency, Xf_test[no_recency])

forward_comparison = pd.DataFrame(rows).T.round(3)
forward_comparison

,Accuracy,Precision,Recall,F1,AUC
Logistic Regression,0.740,0.753,0.801,0.777,0.809
XGBoost,0.756,0.755,0.840,0.795,0.819
Neural Network (MLP),0.750,0.750,0.837,0.791,0.816
Random Forest,0.762,0.756,0.855,0.803,0.818
Random Forest without Recency,0.721,0.710,0.854,0.775,0.764


<br>

### Forward-Looking Results

- **Random Forest is the best model:** 76.2% accuracy, 85.5% recall, F1 0.803, AUC 0.818.

- **All four models are close**, so the features matter more than the algorithm.

- **Recency is the strongest signal.** Without it, the same Random Forest drops to 72.1% accuracy and 0.764 AUC.

- **Baseline:** always predicting churn would score 56.5%.

<br>

<br>

---

## Step 15: Is the Result Real?

Two checks before trusting the numbers.

1. **Overfitting check:** does the model score the same on customers it has never seen?

2. **Out-of-time test:** does it still work when trained on an earlier period and tested on a later one?

<br>

<br>

### Check 1: Overfitting

Compare accuracy on the training customers with accuracy on the test customers.

<br>

In [ ]:
rf_forward = fitted["Random Forest"]

train_acc = accuracy_score(yf_train, rf_forward.predict(Xf_train))
test_acc = accuracy_score(yf_test, rf_forward.predict(Xf_test))

print(f"Train accuracy: {train_acc:.3f}")
print(f"Test accuracy:  {test_acc:.3f}")

Train accuracy: 0.766
Test accuracy:  0.762


<br>

Then repeat the test on five different splits (5-fold cross-validation).

<br>

In [ ]:
cv_auc = cross_val_score(
    RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42),
    late[fwd_cols],
    late["Churned"],
    cv=StratifiedKFold(5, shuffle=True, random_state=0),
    scoring="roc_auc",
)

print(f"Average AUC over 5 folds: {cv_auc.mean():.3f}")
print(f"Lowest fold:  {cv_auc.min():.3f}")
print(f"Highest fold: {cv_auc.max():.3f}")

Average AUC over 5 folds: 0.797
Lowest fold:  0.793
Highest fold: 0.802


<br>

**Result:** no overfitting.

- Train accuracy 76.6% vs test accuracy 76.2%.

- Cross-validated AUC is 0.797, so **about 0.80 is the honest AUC**. The 0.818 in Step 14 is one slightly favourable split.

<br>

<br>

### Check 2: Out-of-Time Test

Train the models at the earlier cutoff, then test them on every customer at the later cutoff.

<br>

In [ ]:
print("Train at cutoff:", cutoff_early.date())
print("Test at cutoff: ", cutoff_late.date(), "on all", len(late), "customers")
print()
print(f"Stopped buying in the training window: {early['Churned'].mean() * 100:.1f}%")
print(f"Stopped buying in the test window:     {late['Churned'].mean() * 100:.1f}%")

Train at cutoff: 2011-06-12
Test at cutoff:  2011-09-10 on all 5258 customers

Stopped buying in the training window: 67.8%
Stopped buying in the test window:     56.4%


<br>

Now train every model on the earlier period and score it on the later one.

<br>

In [ ]:
oot_rows = {}

# Train on the earlier period, score on the later period
for name, model in make_models().items():
    model.fit(early[fwd_cols], early["Churned"])
    oot_rows[name] = scores(late["Churned"], model, late[fwd_cols])

# Simple rule for comparison: flag anyone with no purchase in 90 days
rule = (late["Recency"] > 90).astype(int)
oot_rows["Simple rule: no purchase in 90 days"] = {
    "Accuracy": accuracy_score(late["Churned"], rule),
    "Precision": precision_score(late["Churned"], rule),
    "Recall": recall_score(late["Churned"], rule),
    "F1": f1_score(late["Churned"], rule),
    "AUC": np.nan,
}

pd.DataFrame(oot_rows).T.round(3)

,Accuracy,Precision,Recall,F1,AUC
Logistic Regression,0.711,0.679,0.927,0.784,0.797
XGBoost,0.703,0.673,0.919,0.777,0.791
Neural Network (MLP),0.706,0.674,0.926,0.780,0.796
Random Forest,0.700,0.666,0.939,0.779,0.791
Simple rule: no purchase in 90 days,0.703,0.709,0.802,0.753,NaN


<br>

**Result:** the ranking holds, the yes/no cut-off does not.

- **AUC stays at 0.79 to 0.80** for every model, so the ranking of customers by risk is stable over time.

- **Accuracy drops to about 70%**, level with the simple rule (70.3%).

- **Why:** churn is seasonal. 67.8% of customers went quiet in the summer training window, but only 56.4% before Christmas. A model trained on summer flags too many customers in autumn.

**Takeaway:** use the model to **rank customers by churn risk (AUC about 0.80)**, and reset the yes/no cut-off for each season.

<br>

<br>

---

## Step 16: The Result in Numbers of Customers

Percentages are hard to picture. This step shows the same results as customer counts.

<br>

<br>

### How Many Leavers the Model Catches

On the 1,052 held-out customers, compared with the simple rule.

<br>

In [ ]:
def counts(y_true, pred):
    tn, fp, fn, tp = confusion_matrix(y_true, pred).ravel()
    return {
        "Flagged as leaving": tp + fp,
        "Correct (really left)": tp,
        "False alarms": fp,
        "Leavers missed": fn,
    }


print("Held-out customers:", len(yf_test))
print("Really stopped buying:", int(yf_test.sum()))
print("Kept buying:", int((1 - yf_test).sum()))

model_pred = fitted["Random Forest"].predict(Xf_test)
rule_pred = (Xf_test["Recency"] > 90).astype(int)

pd.DataFrame(
    {
        "Random Forest": counts(yf_test, model_pred),
        "Simple rule (no purchase in 90 days)": counts(yf_test, rule_pred),
    }
).T

Held-out customers: 1052
Really stopped buying: 594
Kept buying: 458


,Flagged as leaving,Correct (really left),False alarms,Leavers missed
Random Forest,672,508,164,86
Simple rule (no purchase in 90 days),685,494,191,100


<br>

**Result:**

- 594 customers really stopped buying. The model caught **508 of them (85.5%)**.

- It raised 164 false alarms and missed 86 leavers.

- Compared with the simple rule, the model finds **14 more leavers with 27 fewer false alarms**. A real but modest gain.

<br>

<br>

### Ranking Customers by Risk

Score all 5,258 customers and split them into five equal groups, from safest to riskiest.

<br>

In [ ]:
# Score every customer with the model trained on the earlier period
rf_oot = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
rf_oot.fit(early[fwd_cols], early["Churned"])

scored = late.copy()
scored["risk"] = rf_oot.predict_proba(late[fwd_cols])[:, 1]

# Each customer's spend in the 12 months before the cutoff
year_start = cutoff_late - pd.Timedelta(days=365)
last_year = data[(data["InvoiceDate"] > year_start) & (data["InvoiceDate"] <= cutoff_late)]
spend = (last_year["Quantity"] * last_year["Price"]).groupby(last_year["Customer ID"]).sum()
scored["spend_12m"] = spend.reindex(scored.index).fillna(0)

# Five equal groups, from safest to riskiest
scored["Risk group"] = pd.qcut(
    scored["risk"].rank(method="first"),
    5,
    labels=["Safest 20%", "Low", "Middle", "High", "Riskiest 20%"],
)

<br>

Summarise each risk group in one table.

<br>

In [ ]:
groups = scored.groupby("Risk group", observed=True)
total_spend = scored["spend_12m"].sum()

risk_table = pd.DataFrame(
    {
        "Customers": groups.size(),
        "Stopped buying": groups["Churned"].sum(),
        "% stopped buying": (groups["Churned"].mean() * 100).round(1),
        "Spend in prior 12 months (£)": groups["spend_12m"].sum().round(0).astype(int),
        "% of total spend": (groups["spend_12m"].sum() / total_spend * 100).round(1),
    }
)

risk_table.sort_index(ascending=False)

,Customers,Stopped buying,% stopped buying,Spend in prior 12 months (£),% of total spend
Risk group,,,,,
Riskiest 20%,1052,903,85.8,103026,1.2
High,1051,787,74.9,380261,4.4
Middle,1052,665,63.2,803353,9.4
Low,1051,439,41.8,1257953,14.7
Safest 20%,1052,174,16.5,6005182,70.2


<br>

**Result:**

- The ranking works: **85.8% of the riskiest group stopped buying**, against 16.5% of the safest group.

- But the riskiest group brings in only 1.2% of the spend, so it is not where the money is.

<br>

<br>

### Where the Model Is Most Valuable: High-Value Customers

Look only at the top 25% of customers by spend.

<br>

In [ ]:
# High-value = top 25% of customers by spend in the prior 12 months
threshold = scored["spend_12m"].quantile(0.75)
high_value = scored[scored["spend_12m"] >= threshold].sort_values("risk", ascending=False)

share_of_spend = high_value["spend_12m"].sum() / total_spend * 100

print("High-value customers:", len(high_value))
print(f"Share of total spend: {share_of_spend:.1f}%")
print(f"Stopped buying: {high_value['Churned'].mean() * 100:.1f}%")

High-value customers: 1315
Share of total spend: 82.6%
Stopped buying: 21.1%


<br>

Within the high-value customers, compare the model's riskiest 20% with its safest 20%.

<br>

In [ ]:
k = len(high_value) // 5
riskiest = high_value.iloc[:k]
safest = high_value.iloc[-k:]

high_value_table = pd.DataFrame(
    [
        [
            len(g),
            int(g["Churned"].sum()),
            round(g["Churned"].mean() * 100, 1),
            int(round(g["spend_12m"].sum())),
        ]
        for g in (riskiest, safest)
    ],
    index=[
        "Riskiest 20% of high-value customers",
        "Safest 20% of high-value customers",
    ],
    columns=[
        "Customers",
        "Stopped buying",
        "% stopped buying",
        "Spend in prior 12 months (£)",
    ],
)

high_value_table

,Customers,Stopped buying,% stopped buying,Spend in prior 12 months (£)
Riskiest 20% of high-value customers,263,122,46.4,853812
Safest 20% of high-value customers,263,12,4.6,3834560


<br>

**Result:**

- High-value customers bring in **82.6% of the spend**, and 21.1% of them stopped buying.

- The model's riskiest 20% of them (**263 customers, about £854K of prior-year spend**) stopped buying **46.4%** of the time.

- Its safest 20% stopped buying only **4.6%** of the time.

These 263 customers are the ones worth contacting first.

<br>

<br>

---

## What Comes Next

The API and the live app (Steps 11 and 12) still use the snapshot Random Forest from Part 1. Moving them to the forward-looking model from Part 2 would make the deployed version answer the real business question.

<br>